In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# AgriHerd AI Model Training\n",
    "\n",
    "**Assignment:** BIDT3110 - Digital Strategy for Business Transformation\n",
    "\n",
    "**Objective:** Train a YOLOv8 object detection model to detect 30 crop diseases from leaf images.\n",
    "\n",
    "**Dataset:** Multicrop Disease Dataset (Roboflow) - 2,178 images across 30 classes.\n",
    "\n",
    "**Environment:** Google Colab with T4 GPU"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "!pip install ultralytics\n",
    "from ultralytics import YOLO\n",
    "import os"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1: Upload Dataset\n",
    "\n",
    "The dataset was downloaded as a ZIP file from Roboflow and uploaded to Colab."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import zipfile\n",
    "os.makedirs(\"dataset\", exist_ok=True)\n",
    "with zipfile.ZipFile(\"agriherd_data.zip\", 'r') as z:\n",
    "    z.extractall(\"dataset/\")\n",
    "print(\"Dataset extracted\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2: Fix data.yaml paths\n",
    "\n",
    "YOLO needs absolute paths inside the YAML file."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "yaml_path = \"dataset/data.yaml\"\n",
    "with open(yaml_path) as f:\n",
    "    lines = f.readlines()\n",
    "with open(yaml_path, \"w\") as f:\n",
    "    for line in lines:\n",
    "        if line.startswith(\"path:\"):\n",
    "            f.write(\"path: /content/dataset\\n\")\n",
    "        else:\n",
    "            f.write(line)\n",
    "print(\"data.yaml fixed\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3: Train Model\n",
    "\n",
    "Training parameters:\n",
    "- 20 epochs\n",
    "- Image size 640x640\n",
    "- Batch size 16\n",
    "- Base model: YOLOv8n (nano - lightweight, fast)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "model = YOLO(\"yolov8n.pt\")\n",
    "results = model.train(\n",
    "    data=\"dataset/data.yaml\",\n",
    "    epochs=20,\n",
    "    imgsz=640,\n",
    "    batch=16,\n",
    "    name=\"agriherd_model\",\n",
    "    patience=5\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4: Validate Model\n",
    "\n",
    "The best model achieved:\n",
    "- mAP50: 69.3%\n",
    "- Precision: 72.4%\n",
    "- Recall: 65.6%\n",
    "\n",
    "These are the final scores after 20 epochs of training."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "best_model = YOLO(\"runs/detect/agriherd_model/weights/best.pt\")\n",
    "metrics = best_model.val(data=\"dataset/data.yaml\", split=\"test\")\n",
    "print(f\"mAP50: {metrics.box.map50}\")\n",
    "print(f\"Precision: {metrics.box.mp}\")\n",
    "print(f\"Recall: {metrics.box.mr}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5: Test Single Image Prediction\n",
    "\n",
    "This demonstrates the model in action."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import random\n",
    "val_dir = \"dataset/images/val\"\n",
    "sample = random.choice(os.listdir(val_dir))\n",
    "result = best_model(os.path.join(val_dir, sample))\n",
    "print(f\"Image: {sample}\")\n",
    "for box in result[0].boxes:\n",
    "    print(f\"  {best_model.names[int(box.cls[0])]}: {float(box.conf[0])*100:.1f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6: Download Best Model\n",
    "\n",
    "Download `runs/detect/agriherd_model/weights/best.pt` for use in the AgriHerd system."
   ]
  }
 ],
 "metadata": {
  "colab": {
   "name": "AgriHerd_Model_Training.ipynb",
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}